In [2]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor

In [24]:
from sklearn.decomposition import PCA

In [4]:
train_df = pd.read_csv(
    "/content/train.csv"
)

train_df.shape

(75000, 4)

In [5]:
sample_df = train_df.sample(
    5000,
    random_state=42
).reset_index(drop=True)

sample_df.shape

(5000, 4)

In [6]:
X_train_text, X_valid_text, y_train, y_valid = train_test_split(
    sample_df["catalog_content"],
    sample_df["price"],
    test_size=0.2,
    random_state=42
)

In [7]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_valid_tfidf = tfidf.transform(
    X_valid_text
)

print(X_train_tfidf.shape)
print(X_valid_tfidf.shape)

(4000, 5000)
(1000, 5000)


In [8]:
def extract_quantity_features(text):
    text = str(text).lower()

    ounce = re.search(r'(\d+\.?\d*)\s*(oz|ounce)', text)
    pound = re.search(r'(\d+\.?\d*)\s*(lb|pound)', text)
    pack = re.search(r'pack of (\d+)', text)
    serving = re.search(r'(\d+)\s*servings', text)
    count = re.search(r'(\d+)\s*count', text)

    return pd.Series([
        float(ounce.group(1)) if ounce else 0,
        float(pound.group(1)) if pound else 0,
        float(pack.group(1)) if pack else 0,
        float(serving.group(1)) if serving else 0,
        float(count.group(1)) if count else 0
    ])


quantity_features = sample_df[
    "catalog_content"
].apply(extract_quantity_features)

quantity_features.columns = [
    "ounce_feature",
    "pound_feature",
    "pack_feature",
    "serving_feature",
    "count_feature"
]

quantity_features.head()

,ounce_feature,pound_feature,pack_feature,serving_feature,count_feature
0,0.0,0.0,12.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,12.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,25.4,0.0,12.0,62.0,0.0


In [9]:
quantity_train = quantity_features.loc[
    X_train_text.index
]

quantity_valid = quantity_features.loc[
    X_valid_text.index
]

print(quantity_train.shape)
print(quantity_valid.shape)

(4000, 5)
(1000, 5)


In [10]:
quantity_train_sparse = csr_matrix(
    quantity_train.values
)

quantity_valid_sparse = csr_matrix(
    quantity_valid.values
)

In [11]:
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import torch
import timm

from torchvision import transforms

In [12]:
model = timm.create_model(
    "resnet50",
    pretrained=True,
    num_classes=0
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act1): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
      (act2): ReLU(inplace=True)
      (aa): Identity()
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     

In [13]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [14]:
def get_image_embedding(url):
    try:
        response = requests.get(
            url,
            timeout=10
        )

        image = Image.open(
            BytesIO(response.content)
        ).convert("RGB")

        image = transform(image)

        image = image.unsqueeze(0)

        with torch.no_grad():
            embedding = model(image)

        return embedding.squeeze().numpy()

    except:
        return np.zeros(2048)

In [15]:
X_train_images = sample_df.loc[
    X_train_text.index,
    "image_link"
]

X_valid_images = sample_df.loc[
    X_valid_text.index,
    "image_link"
]

In [16]:
train_embeddings = []

for url in tqdm(X_train_images):
    train_embeddings.append(
        get_image_embedding(url)
    )

train_embeddings = np.array(
    train_embeddings
)

100%|██████████| 4000/4000 [21:24<00:00,  3.11it/s]


In [17]:
valid_embeddings = []

for url in tqdm(X_valid_images):
    valid_embeddings.append(
        get_image_embedding(url)
    )

valid_embeddings = np.array(
    valid_embeddings
)

100%|██████████| 1000/1000 [04:10<00:00,  3.99it/s]


In [18]:
print(train_embeddings.shape)
print(valid_embeddings.shape)

(4000, 2048)
(1000, 2048)


In [25]:
pca = PCA(n_components=100)

train_embeddings_pca = pca.fit_transform(
    train_embeddings
)

valid_embeddings_pca = pca.transform(
    valid_embeddings
)

print(train_embeddings_pca.shape)
print(valid_embeddings_pca.shape)

train_image_embeddings = csr_matrix(
    train_embeddings_pca
)

valid_image_embeddings = csr_matrix(
    valid_embeddings_pca
)

(4000, 100)
(1000, 100)


In [26]:
X_train_final = hstack([
    X_train_tfidf,
    quantity_train_sparse,
    train_image_embeddings
])

X_valid_final = hstack([
    X_valid_tfidf,
    quantity_valid_sparse,
    valid_image_embeddings
])

print(X_train_final.shape)
print(X_valid_final.shape)

(4000, 5105)
(1000, 5105)


In [27]:
multimodal_model = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=50,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

multimodal_model.fit(
    X_train_final,
    y_train
)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.052988 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 87124
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 1941
[LightGBM] [Info] Start training from score 24.681082
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

LGBMRegressor(colsample_bytree=0.8, learning_rate=0.03, max_depth=10,
              n_estimators=800, num_leaves=50, random_state=42, subsample=0.8)

In [28]:
multimodal_predictions = multimodal_model.predict(
    X_valid_final
)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [29]:
mae = mean_absolute_error(
    y_valid,
    multimodal_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        multimodal_predictions
    )
)

r2 = r2_score(
    y_valid,
    multimodal_predictions
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 15.371983118885613
RMSE: 32.10970370652931
R2 Score: 0.15286190729704996
